In [2]:
%pip install pillow


Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install PySide6

   ---------------------------------------- 0.0/578.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/578.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/578.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/578.4 kB ? eta -:--:--
   ------------------ --------------------- 262.1/578.4 kB ? eta -:--:--
   ---------------------------------------- 578.4/578.4 kB 1.3 MB/s  0:00:00
   ---------------------------------------- 0.0/168.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/168.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/168.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/168.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/168.8 MB ? eta -:--:--
   ---------------------------------------- 0.8/168.8 MB 798.0 kB/s eta 0:03:31
   ---------------------------------------- 1.3/168.8 MB 1.1 MB/s eta 0:02:30
   ---------------------------------------- 1.6/168.8

In [1]:
# Install dependencies if necessary
# %pip install opencv-python numpy "mediapipe<0.10.21" PySide6

import sys
import cv2
import numpy as np
import time
import os
import threading
from pathlib import Path

import mediapipe as mp
from PySide6.QtWidgets import (QApplication, QMainWindow, QWidget, QVBoxLayout, 
                               QHBoxLayout, QLabel, QLineEdit, QPushButton, QMessageBox)
from PySide6.QtCore import QTimer, Qt, Slot
from PySide6.QtGui import QImage, QPixmap

# Core data processing functions matching the main pipeline format
MAX_SEQ_LENGTH = 120
FEATURES = 141
FPS = 30
WIDTH = 640
HEIGHT = 480

mp_holistic = mp.solutions.holistic
mp_draw = mp.solutions.drawing_utils
POSE = [0, 11, 12, 13, 14]

def normalize_frame(vec):
    vec = vec.copy()
    pose = vec[:15].reshape(5, 3)
    lh = vec[15:78].reshape(21, 3)
    rh = vec[78:].reshape(21, 3)

    if np.any(pose):
        ls = pose[1]
        rs = pose[2]
        center = (ls + rs) / 2
        dist = np.linalg.norm(ls - rs)
        if dist > 1e-6:
            pose = (pose - center) / dist
            if np.any(lh): lh = (lh - center) / dist
            if np.any(rh): rh = (rh - center) / dist

    return np.concatenate([pose.flatten(), lh.flatten(), rh.flatten()])

def extract(results):
    pose = []
    if results.pose_landmarks:
        for idx in POSE:
            lm = results.pose_landmarks.landmark[idx]
            pose.extend([lm.x, lm.y, lm.z])
    else: pose = [0] * 15

    lh = []
    if results.left_hand_landmarks:
        for lm in results.left_hand_landmarks.landmark:
            lh.extend([lm.x, lm.y, lm.z])
    else: lh = [0] * 63

    rh = []
    if results.right_hand_landmarks:
        for lm in results.right_hand_landmarks.landmark:
            rh.extend([lm.x, lm.y, lm.z])
    else: rh = [0] * 63

    return normalize_frame(np.array(pose + lh + rh, dtype=np.float32))

def pad(seq):
    if len(seq) < MAX_SEQ_LENGTH:
        seq = np.vstack([seq, np.zeros((MAX_SEQ_LENGTH - len(seq), FEATURES))])
    return seq[:MAX_SEQ_LENGTH]

def reconstruct(data, outfile):
    writer = cv2.VideoWriter(str(outfile), cv2.VideoWriter_fourcc(*"mp4v"), FPS, (WIDTH, HEIGHT))
    for i, frame in enumerate(data):
        canvas = np.full((HEIGHT, WIDTH, 3), 255, dtype=np.uint8)
        if np.all(frame == 0):
            cv2.putText(canvas, "PAD", (280, 220), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 2)
            writer.write(canvas)
            continue

        pose = frame[:15].reshape(5, 3)
        lh = frame[15:78].reshape(21, 3)
        rh = frame[78:].reshape(21, 3)

        def draw(points, color):
            for p in points:
                if p[0] == 0 and p[1] == 0 and p[2] == 0:
                    continue
                x = int((p[0] * 0.25 + 0.5) * WIDTH)
                y = int((p[1] * 0.25 + 0.5) * HEIGHT)
                cv2.circle(canvas, (x, y), 3, color, -1)

        draw(pose, (0, 0, 255))
        draw(lh, (0, 255, 0))
        draw(rh, (255, 255, 0))

        cv2.putText(canvas, f"Frame {i}", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, .7, (0, 0, 0), 2)
        writer.write(canvas)
    writer.release()

class App(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Bridging Silence - Webcam Collector (PySide6)")
        
        self.vid = None
        
        self.is_recording = False
        self.holo = mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5)

        self.init_ui()
        
        self.timer = QTimer(self)
        self.timer.timeout.connect(self.update_frame)
        self.delay = 15
        self.timer.start(self.delay)
        
    def init_ui(self):
        central_widget = QWidget()
        self.setCentralWidget(central_widget)
        main_layout = QVBoxLayout(central_widget)
        
        control_layout = QHBoxLayout()
        
        control_layout.addWidget(QLabel("Word:"))
        self.word_entry = QLineEdit("HELLO")
        self.word_entry.setFixedWidth(100)
        control_layout.addWidget(self.word_entry)
        
        control_layout.addWidget(QLabel("Samples:"))
        self.samples_entry = QLineEdit("3")
        self.samples_entry.setFixedWidth(50)
        control_layout.addWidget(self.samples_entry)
        
        control_layout.addWidget(QLabel("Frames:"))
        self.frames_entry = QLineEdit("90")
        self.frames_entry.setFixedWidth(50)
        control_layout.addWidget(self.frames_entry)
        
        control_layout.addWidget(QLabel("Camera:"))
        self.cam_entry = QLineEdit("0")
        self.cam_entry.setFixedWidth(40)
        control_layout.addWidget(self.cam_entry)
        
        self.start_cam_btn = QPushButton("Open Camera")
        self.start_cam_btn.clicked.connect(self.open_camera)
        control_layout.addWidget(self.start_cam_btn)

        self.btn_record = QPushButton("Start Rec")
        self.btn_record.setEnabled(False)
        self.btn_record.clicked.connect(self.start_recording)
        control_layout.addWidget(self.btn_record)
        
        self.btn_stop = QPushButton("Stop Rec")
        self.btn_stop.setEnabled(False)
        self.btn_stop.clicked.connect(self.stop_recording)
        control_layout.addWidget(self.btn_stop)
        
        main_layout.addLayout(control_layout)
        
        self.video_label = QLabel()
        self.video_label.setFixedSize(WIDTH, HEIGHT)
        self.video_label.setStyleSheet("background-color: black;")
        main_layout.addWidget(self.video_label)
        
        self.status_label = QLabel("Ready. Adjust Camera Index and click Open Camera.")
        main_layout.addWidget(self.status_label)
        
    def open_camera(self):
        if self.vid is not None:
            self.vid.release()
            self.vid = None
            self.start_cam_btn.setText("Open Camera")
            self.btn_record.setEnabled(False)
            self.btn_stop.setEnabled(False)
            self.status_label.setText("Camera closed.")
            self.video_label.clear()
            self.video_label.setStyleSheet("background-color: black;")
            return
            
        cam_idx = 0
        try:
            cam_idx = int(self.cam_entry.text())
        except ValueError:
            pass
            
        if os.name == 'nt':
            self.vid = cv2.VideoCapture(cam_idx, cv2.CAP_DSHOW)
        else:
            self.vid = cv2.VideoCapture(cam_idx)
            
        if not self.vid.isOpened():
            self.vid = cv2.VideoCapture(cam_idx)
            
        if self.vid.isOpened():
            self.start_cam_btn.setText("Close Camera")
            self.btn_record.setEnabled(True)
            self.status_label.setText(f"Camera {cam_idx} opened.")
        else:
            self.status_label.setText(f"Failed to open camera {cam_idx}")
            self.vid = None

    def start_recording(self):
        if not self.vid or not self.vid.isOpened():
            QMessageBox.critical(self, "Error", "Camera is not opened.")
            return
            
        self.word = self.word_entry.text().strip()
        if not self.word:
            QMessageBox.critical(self, "Error", "Please enter a word.")
            return
            
        try:
            self.num_samples = int(self.samples_entry.text())
            self.frames_per_sample = int(self.frames_entry.text())
        except ValueError:
            QMessageBox.critical(self, "Error", "Samples and Frames must be integers.")
            return

        self.btn_record.setEnabled(False)
        self.btn_stop.setEnabled(True)
        self.is_recording = True
        self.current_sample = 0
        self.countdown = 3
        self.wait_timer = 0
        self.last_time = time.time()

    def stop_recording(self):
        self.is_recording = False
        self.btn_record.setEnabled(True)
        self.btn_stop.setEnabled(False)
        self.status_label.setText("Recording stopped.")

    def save_sample_task(self, frames, seq, sample_idx):
        sample_name = f"sample_{int(time.time())}"
        
        WEBCAM_DATA_DIR = "webcam_data"
        WEBCAM_OUT_DIR = "processed_webcam_data"
        
        raw_root = Path(WEBCAM_DATA_DIR) / self.word
        land_root = Path(WEBCAM_OUT_DIR) / "landmarks" / self.word
        prev_root = Path(WEBCAM_OUT_DIR) / "preview" / self.word
        
        raw_root.mkdir(parents=True, exist_ok=True)
        land_root.mkdir(parents=True, exist_ok=True)
        prev_root.mkdir(parents=True, exist_ok=True)
        
        raw_vid_path = raw_root / f"{sample_name}.mp4"
        writer = cv2.VideoWriter(str(raw_vid_path), cv2.VideoWriter_fourcc(*"mp4v"), FPS, (WIDTH, HEIGHT))
        for f in frames: writer.write(f)
        writer.release()
        
        seq_np = np.array(seq)
        seq_np = pad(seq_np)
        
        npy_path = land_root / f"{sample_name}.npy"
        np.save(npy_path, seq_np)
        
        prev_path = prev_root / f"{sample_name}.mp4"
        reconstruct(seq_np, prev_path)
        
        print(f"Saved {sample_name} for word '{self.word}'")

    def save_sample(self):
        frames_copy = list(self.frames)
        seq_copy = list(self.seq)
        
        threading.Thread(target=self.save_sample_task, args=(frames_copy, seq_copy, self.current_sample)).start()
        
        self.current_sample += 1
        if self.current_sample < self.num_samples:
            self.wait_timer = 1
            self.last_time = time.time()
            self.status_label.setText("Waiting before next sample...")
        else:
            self.is_recording = False
            self.btn_record.setEnabled(True)
            self.btn_stop.setEnabled(False)
            self.status_label.setText("Collection complete!")

    def set_image(self, frame):
        rgb_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        h, w, ch = rgb_image.shape
        bytes_per_line = ch * w
        q_img = QImage(rgb_image.data, w, h, bytes_per_line, QImage.Format_RGB888)
        self.video_label.setPixmap(QPixmap.fromImage(q_img))

    def update_frame(self):
        if self.vid is not None and self.vid.isOpened():
            ret, frame = self.vid.read()
            if ret:
                frame = cv2.resize(frame, (WIDTH, HEIGHT))
                
                if self.is_recording:
                    if self.countdown > 0:
                        if time.time() - self.last_time >= 1.0:
                            self.countdown -= 1
                            self.last_time = time.time()
                        
                        display_frame = frame.copy()
                        cv2.putText(display_frame, f"Starting {self.current_sample+1} in {self.countdown}...", 
                                    (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                        
                        self.set_image(display_frame)
                        
                        if self.countdown == 0:
                            self.status_label.setText(f"Recording: {self.word} (Sample {self.current_sample+1}/{self.num_samples})")
                            self.frames = []
                            self.seq = []
                    
                    elif self.wait_timer > 0:
                        if time.time() - self.last_time >= 1.0:
                            self.wait_timer -= 1
                            self.last_time = time.time()
                            if self.wait_timer == 0:
                                self.countdown = 3
                        
                        display_frame = frame.copy()
                        cv2.putText(display_frame, f"Wait...", 
                                    (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 165, 0), 2)
                        
                        self.set_image(display_frame)
                        
                    else:
                        self.frames.append(frame)
                        rgb_for_mp = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                        res = self.holo.process(rgb_for_mp)
                        self.seq.append(extract(res))
                        
                        display_frame = frame.copy()
                        cv2.putText(display_frame, f"Recording [{len(self.frames)}/{self.frames_per_sample}]", 
                                    (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                                    
                        self.set_image(display_frame)
                        
                        if len(self.frames) >= self.frames_per_sample:
                            self.save_sample()
                else:
                    self.set_image(frame)

    def closeEvent(self, event):
        if self.vid is not None:
            self.vid.release()
        self.holo.close()
        event.accept()

def main():
    app = QApplication.instance()
    if not app:
        app = QApplication(sys.argv)
    window = App()
    window.show()
    app.exec()

if __name__ == '__main__':
    main()


d:\BRIDGING SILENCE DATA COLLECTION PIPELINE\.venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Saved sample_1782133037 for word 'HELLO'


KeyboardInterrupt: 

In [2]:
# Install dependencies if necessary
# %pip install opencv-python numpy "mediapipe<0.10.21" PySide6

import sys
import cv2
import numpy as np
import time
import os
import threading
from pathlib import Path

import mediapipe as mp
from PySide6.QtWidgets import (QApplication, QMainWindow, QWidget, QVBoxLayout,
                               QHBoxLayout, QLabel, QLineEdit, QPushButton,
                               QMessageBox, QFrame, QSizePolicy, QGraphicsDropShadowEffect)
from PySide6.QtCore import QTimer, Qt, Slot, QPropertyAnimation, QEasingCurve, QRect, Property, QObject
from PySide6.QtGui import QImage, QPixmap, QFont, QColor, QPainter, QPen, QBrush, QPalette, QLinearGradient

# ── Core data processing (UNCHANGED) ──────────────────────────────────────────
MAX_SEQ_LENGTH = 120
FEATURES = 141
FPS = 30
WIDTH = 640
HEIGHT = 480

mp_holistic = mp.solutions.holistic
mp_draw = mp.solutions.drawing_utils
POSE = [0, 11, 12, 13, 14]

def normalize_frame(vec):
    vec = vec.copy()
    pose = vec[:15].reshape(5, 3)
    lh = vec[15:78].reshape(21, 3)
    rh = vec[78:].reshape(21, 3)
    if np.any(pose):
        ls = pose[1]; rs = pose[2]
        center = (ls + rs) / 2
        dist = np.linalg.norm(ls - rs)
        if dist > 1e-6:
            pose = (pose - center) / dist
            if np.any(lh): lh = (lh - center) / dist
            if np.any(rh): rh = (rh - center) / dist
    return np.concatenate([pose.flatten(), lh.flatten(), rh.flatten()])

def extract(results):
    pose = []
    if results.pose_landmarks:
        for idx in POSE:
            lm = results.pose_landmarks.landmark[idx]
            pose.extend([lm.x, lm.y, lm.z])
    else: pose = [0] * 15
    lh = []
    if results.left_hand_landmarks:
        for lm in results.left_hand_landmarks.landmark:
            lh.extend([lm.x, lm.y, lm.z])
    else: lh = [0] * 63
    rh = []
    if results.right_hand_landmarks:
        for lm in results.right_hand_landmarks.landmark:
            rh.extend([lm.x, lm.y, lm.z])
    else: rh = [0] * 63
    return normalize_frame(np.array(pose + lh + rh, dtype=np.float32))

def pad(seq):
    if len(seq) < MAX_SEQ_LENGTH:
        seq = np.vstack([seq, np.zeros((MAX_SEQ_LENGTH - len(seq), FEATURES))])
    return seq[:MAX_SEQ_LENGTH]

def reconstruct(data, outfile):
    writer = cv2.VideoWriter(str(outfile), cv2.VideoWriter_fourcc(*"mp4v"), FPS, (WIDTH, HEIGHT))
    for i, frame in enumerate(data):
        canvas = np.full((HEIGHT, WIDTH, 3), 255, dtype=np.uint8)
        if np.all(frame == 0):
            cv2.putText(canvas, "PAD", (280, 220), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 2)
            writer.write(canvas); continue
        pose = frame[:15].reshape(5, 3)
        lh = frame[15:78].reshape(21, 3)
        rh = frame[78:].reshape(21, 3)
        def draw(points, color):
            for p in points:
                if p[0] == 0 and p[1] == 0 and p[2] == 0:
                    continue
                x = int((p[0] * 0.25 + 0.5) * WIDTH)
                y = int((p[1] * 0.25 + 0.5) * HEIGHT)
                cv2.circle(canvas, (x, y), 3, color, -1)
        draw(pose, (0, 0, 255)); draw(lh, (0, 255, 0)); draw(rh, (255, 255, 0))
        cv2.putText(canvas, f"Frame {i}", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, .7, (0, 0, 0), 2)
        writer.write(canvas)
    writer.release()

# ── Design tokens ──────────────────────────────────────────────────────────────
BG_DEEP     = "#0A0D14"   # near-black with blue cast
BG_CARD     = "#111622"   # card surfaces
BG_RAISED   = "#161D2E"   # slightly lifted panels
BORDER      = "#1E2A40"   # subtle borders
CYAN        = "#00D4FF"   # primary accent — electric cyan
CYAN_DIM    = "#0A4A5C"   # muted cyan for inactive
WHITE       = "#EDF2F7"   # primary text
GREY        = "#6B7A99"   # secondary text / labels
RED_REC     = "#FF3B3B"   # recording indicator
AMBER       = "#FFB340"   # countdown / warning
GREEN_OK    = "#00E676"   # complete / success

MONO = "Courier New"
SANS = "Segoe UI"  # falls back gracefully on all platforms

# ── Reusable styled widgets ────────────────────────────────────────────────────
class Divider(QFrame):
    def __init__(self, parent=None):
        super().__init__(parent)
        self.setFrameShape(QFrame.HLine)
        self.setFixedHeight(1)
        self.setStyleSheet(f"background-color: {BORDER}; border: none;")

class FieldLabel(QLabel):
    def __init__(self, text, parent=None):
        super().__init__(text.upper(), parent)
        self.setFont(QFont(MONO, 8))
        self.setStyleSheet(f"color: {GREY}; letter-spacing: 2px;")

class DataLabel(QLabel):
    """Mono value display."""
    def __init__(self, text="—", parent=None):
        super().__init__(text, parent)
        self.setFont(QFont(MONO, 11, QFont.Bold))
        self.setStyleSheet(f"color: {CYAN};")

class StyledInput(QLineEdit):
    def __init__(self, text="", width=None, parent=None):
        super().__init__(text, parent)
        if width: self.setFixedWidth(width)
        self.setFont(QFont(MONO, 11))
        self.setStyleSheet(f"""
            QLineEdit {{
                background: {BG_DEEP};
                color: {WHITE};
                border: 1px solid {BORDER};
                border-radius: 4px;
                padding: 6px 10px;
            }}
            QLineEdit:focus {{
                border: 1px solid {CYAN};
            }}
        """)

class PrimaryButton(QPushButton):
    def __init__(self, text, parent=None):
        super().__init__(text, parent)
        self.setFont(QFont(SANS, 10, QFont.Bold))
        self.setFixedHeight(36)
        self.setStyleSheet(f"""
            QPushButton {{
                background: {CYAN};
                color: {BG_DEEP};
                border: none;
                border-radius: 4px;
                padding: 0 18px;
                letter-spacing: 1px;
            }}
            QPushButton:hover {{
                background: #33DDFF;
            }}
            QPushButton:pressed {{
                background: #0099CC;
            }}
            QPushButton:disabled {{
                background: {CYAN_DIM};
                color: {GREY};
            }}
        """)

class DangerButton(QPushButton):
    def __init__(self, text, parent=None):
        super().__init__(text, parent)
        self.setFont(QFont(SANS, 10, QFont.Bold))
        self.setFixedHeight(36)
        self.setStyleSheet(f"""
            QPushButton {{
                background: transparent;
                color: {RED_REC};
                border: 1px solid {RED_REC};
                border-radius: 4px;
                padding: 0 18px;
                letter-spacing: 1px;
            }}
            QPushButton:hover {{
                background: rgba(255,59,59,0.12);
            }}
            QPushButton:pressed {{
                background: rgba(255,59,59,0.25);
            }}
            QPushButton:disabled {{
                color: {GREY};
                border-color: {BORDER};
            }}
        """)

class GhostButton(QPushButton):
    def __init__(self, text, parent=None):
        super().__init__(text, parent)
        self.setFont(QFont(SANS, 10))
        self.setFixedHeight(36)
        self.setStyleSheet(f"""
            QPushButton {{
                background: transparent;
                color: {GREY};
                border: 1px solid {BORDER};
                border-radius: 4px;
                padding: 0 18px;
            }}
            QPushButton:hover {{
                color: {WHITE};
                border-color: {GREY};
            }}
            QPushButton:pressed {{
                background: {BG_RAISED};
            }}
            QPushButton:checked {{
                color: {AMBER};
                border-color: {AMBER};
            }}
        """)

class RecordingPulse(QWidget):
    """Animated REC dot — the signature element."""
    def __init__(self, parent=None):
        super().__init__(parent)
        self.setFixedSize(14, 14)
        self._opacity = 1.0
        self._active = False
        self._pulse_timer = QTimer(self)
        self._pulse_timer.timeout.connect(self._tick)
        self._direction = -1

    def set_active(self, active):
        self._active = active
        if active:
            self._opacity = 1.0
            self._pulse_timer.start(40)
        else:
            self._pulse_timer.stop()
            self._opacity = 0.0
            self.update()

    def _tick(self):
        self._opacity += self._direction * 0.06
        if self._opacity <= 0.15:
            self._direction = 1
        elif self._opacity >= 1.0:
            self._direction = -1
        self.update()

    def paintEvent(self, event):
        p = QPainter(self)
        p.setRenderHint(QPainter.Antialiasing)
        color = QColor(RED_REC)
        color.setAlphaF(self._opacity if self._active else 0.0)
        p.setBrush(QBrush(color))
        p.setPen(Qt.NoPen)
        p.drawEllipse(2, 2, 10, 10)

class ProgressBar(QWidget):
    """Slim custom frame-progress bar."""
    def __init__(self, parent=None):
        super().__init__(parent)
        self.setFixedHeight(6)
        self._value = 0.0   # 0.0 – 1.0

    def set_value(self, v):
        self._value = max(0.0, min(1.0, v))
        self.update()

    def paintEvent(self, event):
        p = QPainter(self)
        p.setRenderHint(QPainter.Antialiasing)
        w, h = self.width(), self.height()
        # track
        p.setBrush(QBrush(QColor(BORDER)))
        p.setPen(Qt.NoPen)
        p.drawRoundedRect(0, 0, w, h, 3, 3)
        # fill
        fill_w = int(w * self._value)
        if fill_w > 0:
            color = QColor(CYAN) if self._value < 1.0 else QColor(GREEN_OK)
            p.setBrush(QBrush(color))
            p.drawRoundedRect(0, 0, fill_w, h, 3, 3)

# ── Main window ────────────────────────────────────────────────────────────────
class App(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Bridging Silence · Data Collector")
        self.setMinimumWidth(820)

        # State (UNCHANGED)
        self.vid = None
        self.is_recording = False
        self.holo = mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5)
        self.frames = []
        self.seq = []
        self.current_sample = 0
        self.num_samples = 3
        self.frames_per_sample = 90
        self.word = "HELLO"
        self.countdown = 0
        self.wait_timer = 0
        self.last_time = time.time()

        self._apply_global_style()
        self._init_ui()

        self.timer = QTimer(self)
        self.timer.timeout.connect(self.update_frame)
        self.timer.start(15)

    def _apply_global_style(self):
        self.setStyleSheet(f"""
            QMainWindow, QWidget {{
                background-color: {BG_DEEP};
                color: {WHITE};
                font-family: '{SANS}';
            }}
            QMessageBox {{
                background-color: {BG_CARD};
                color: {WHITE};
            }}
            QMessageBox QPushButton {{
                background: {CYAN};
                color: {BG_DEEP};
                border: none;
                border-radius: 4px;
                padding: 6px 18px;
                font-weight: bold;
            }}
        """)

    def _section_card(self, layout):
        """Return a card-style inner widget to add to layout."""
        card = QWidget()
        card.setStyleSheet(f"""
            QWidget {{
                background-color: {BG_CARD};
                border: 1px solid {BORDER};
                border-radius: 6px;
            }}
        """)
        layout.addWidget(card)
        return card

    def _init_ui(self):
        root = QWidget()
        self.setCentralWidget(root)
        outer = QHBoxLayout(root)
        outer.setContentsMargins(16, 16, 16, 16)
        outer.setSpacing(14)

        # ── Left column: video ─────────────────────────────────────────────
        left = QVBoxLayout()
        left.setSpacing(8)

        # Header bar
        header = QHBoxLayout()
        title = QLabel("BRIDGING SILENCE")
        title.setFont(QFont(MONO, 12, QFont.Bold))
        title.setStyleSheet(f"color: {WHITE}; letter-spacing: 3px;")
        subtitle = QLabel("Gesture Data Collector v2")
        subtitle.setFont(QFont(MONO, 8))
        subtitle.setStyleSheet(f"color: {GREY};")
        header.addWidget(title)
        header.addStretch()
        header.addWidget(subtitle)
        left.addLayout(header)
        left.addWidget(Divider())

        # Viewfinder
        self.video_label = QLabel()
        self.video_label.setFixedSize(WIDTH, HEIGHT)
        self.video_label.setAlignment(Qt.AlignCenter)
        self.video_label.setStyleSheet(f"""
            background-color: #05080F;
            border: 1px solid {BORDER};
            border-radius: 6px;
        """)
        # Placeholder text
        placeholder = QLabel("NO SIGNAL", self.video_label)
        placeholder.setFont(QFont(MONO, 14))
        placeholder.setStyleSheet(f"color: {BORDER}; background: transparent; border: none;")
        placeholder.setAlignment(Qt.AlignCenter)
        placeholder.setGeometry(0, 0, WIDTH, HEIGHT)
        self._placeholder = placeholder

        left.addWidget(self.video_label)

        # Frame progress bar
        self.frame_progress = ProgressBar()
        left.addWidget(self.frame_progress)

        # Status strip
        status_row = QHBoxLayout()
        self._rec_dot = RecordingPulse()
        self._rec_label = QLabel("IDLE")
        self._rec_label.setFont(QFont(MONO, 9, QFont.Bold))
        self._rec_label.setStyleSheet(f"color: {GREY};")
        status_row.addWidget(self._rec_dot)
        status_row.addSpacing(6)
        status_row.addWidget(self._rec_label)
        status_row.addStretch()
        self.status_label = QLabel("Open a camera source to begin.")
        self.status_label.setFont(QFont(SANS, 9))
        self.status_label.setStyleSheet(f"color: {GREY};")
        status_row.addWidget(self.status_label)
        left.addLayout(status_row)

        outer.addLayout(left)

        # ── Right column: controls ─────────────────────────────────────────
        right = QVBoxLayout()
        right.setSpacing(12)
        right.setContentsMargins(0, 0, 0, 0)

        # ── Camera source card ─────────────────────────────────────────────
        cam_card = QWidget()
        cam_card.setStyleSheet(f"background:{BG_CARD}; border:1px solid {BORDER}; border-radius:6px;")
        cam_inner = QVBoxLayout(cam_card)
        cam_inner.setContentsMargins(16, 14, 16, 14)
        cam_inner.setSpacing(10)

        cam_title = QLabel("CAMERA SOURCE")
        cam_title.setFont(QFont(MONO, 8, QFont.Bold))
        cam_title.setStyleSheet(f"color: {CYAN}; letter-spacing: 2px; border:none; background:transparent;")
        cam_inner.addWidget(cam_title)

        cam_row = QHBoxLayout()
        cam_field = QVBoxLayout()
        cam_field.addWidget(FieldLabel("Index"))
        self.cam_entry = StyledInput("0", width=60)
        cam_field.addWidget(self.cam_entry)
        cam_row.addLayout(cam_field)
        cam_row.addStretch()
        self.start_cam_btn = PrimaryButton("Open Camera")
        self.start_cam_btn.setFixedWidth(130)
        self.start_cam_btn.clicked.connect(self.open_camera)
        cam_row.addWidget(self.start_cam_btn, alignment=Qt.AlignBottom)
        cam_inner.addLayout(cam_row)
        right.addWidget(cam_card)

        # ── Session config card ────────────────────────────────────────────
        cfg_card = QWidget()
        cfg_card.setStyleSheet(f"background:{BG_CARD}; border:1px solid {BORDER}; border-radius:6px;")
        cfg_inner = QVBoxLayout(cfg_card)
        cfg_inner.setContentsMargins(16, 14, 16, 14)
        cfg_inner.setSpacing(10)

        cfg_title = QLabel("SESSION CONFIG")
        cfg_title.setFont(QFont(MONO, 8, QFont.Bold))
        cfg_title.setStyleSheet(f"color: {CYAN}; letter-spacing: 2px; border:none; background:transparent;")
        cfg_inner.addWidget(cfg_title)

        fields_row = QHBoxLayout()
        fields_row.setSpacing(14)

        word_col = QVBoxLayout()
        word_col.addWidget(FieldLabel("Label / Word"))
        self.word_entry = StyledInput("HELLO", width=110)
        word_col.addWidget(self.word_entry)
        fields_row.addLayout(word_col)

        samp_col = QVBoxLayout()
        samp_col.addWidget(FieldLabel("Samples"))
        self.samples_entry = StyledInput("3", width=60)
        samp_col.addWidget(self.samples_entry)
        fields_row.addLayout(samp_col)

        frm_col = QVBoxLayout()
        frm_col.addWidget(FieldLabel("Frames"))
        self.frames_entry = StyledInput("90", width=60)
        frm_col.addWidget(self.frames_entry)
        fields_row.addLayout(frm_col)

        fields_row.addStretch()
        cfg_inner.addLayout(fields_row)
        right.addWidget(cfg_card)

        # ── Live stats card ────────────────────────────────────────────────
        stats_card = QWidget()
        stats_card.setStyleSheet(f"background:{BG_CARD}; border:1px solid {BORDER}; border-radius:6px;")
        stats_inner = QVBoxLayout(stats_card)
        stats_inner.setContentsMargins(16, 14, 16, 14)
        stats_inner.setSpacing(10)

        stats_title = QLabel("LIVE STATS")
        stats_title.setFont(QFont(MONO, 8, QFont.Bold))
        stats_title.setStyleSheet(f"color: {CYAN}; letter-spacing: 2px; border:none; background:transparent;")
        stats_inner.addWidget(stats_title)

        stats_grid = QHBoxLayout()
        stats_grid.setSpacing(20)

        def stat_block(label):
            col = QVBoxLayout()
            col.setSpacing(2)
            lbl = FieldLabel(label)
            val = DataLabel("—")
            col.addWidget(lbl)
            col.addWidget(val)
            stats_grid.addLayout(col)
            return val

        self._stat_word    = stat_block("Word")
        self._stat_sample  = stat_block("Sample")
        self._stat_frame   = stat_block("Frame")
        self._stat_phase   = stat_block("Phase")
        stats_grid.addStretch()
        stats_inner.addLayout(stats_grid)
        right.addWidget(stats_card)

        # ── Record controls card ───────────────────────────────────────────
        rec_card = QWidget()
        rec_card.setStyleSheet(f"background:{BG_CARD}; border:1px solid {BORDER}; border-radius:6px;")
        rec_inner = QVBoxLayout(rec_card)
        rec_inner.setContentsMargins(16, 14, 16, 14)
        rec_inner.setSpacing(10)

        rec_title = QLabel("CAPTURE")
        rec_title.setFont(QFont(MONO, 8, QFont.Bold))
        rec_title.setStyleSheet(f"color: {CYAN}; letter-spacing: 2px; border:none; background:transparent;")
        rec_inner.addWidget(rec_title)

        btn_row = QHBoxLayout()
        self.btn_record = PrimaryButton("▶  Start Recording")
        self.btn_record.setEnabled(False)
        self.btn_record.clicked.connect(self.start_recording)
        self.btn_stop = DangerButton("■  Stop")
        self.btn_stop.setEnabled(False)
        self.btn_stop.clicked.connect(self.stop_recording)
        btn_row.addWidget(self.btn_record)
        btn_row.addWidget(self.btn_stop)
        rec_inner.addLayout(btn_row)

        # Sample progress dots
        self._dot_row = QHBoxLayout()
        self._dot_row.setSpacing(6)
        self._dot_row.addStretch()
        self._sample_dots = []
        for _ in range(10):   # up to 10 visible dots
            dot = QLabel("○")
            dot.setFont(QFont(MONO, 11))
            dot.setStyleSheet(f"color: {BORDER}; border:none; background:transparent;")
            dot.hide()
            self._dot_row.addWidget(dot)
        self._dot_row.addStretch()
        rec_inner.addLayout(self._dot_row)

        right.addWidget(rec_card)
        right.addStretch()

        # Branding footer
        footer = QLabel("BRIDGING SILENCE  ·  ML DATA PIPELINE")
        footer.setFont(QFont(MONO, 7))
        footer.setStyleSheet(f"color: {BORDER};")
        footer.setAlignment(Qt.AlignRight)
        right.addWidget(footer)

        outer.addLayout(right)

    # ── Camera ─────────────────────────────────────────────────────────────────
    def open_camera(self):
        if self.vid is not None:
            self.vid.release()
            self.vid = None
            self.start_cam_btn.setText("Open Camera")
            self.btn_record.setEnabled(False)
            self.btn_stop.setEnabled(False)
            self._set_status("Camera closed.", GREY)
            self.video_label.clear()
            self._placeholder.show()
            return

        cam_idx = 0
        try: cam_idx = int(self.cam_entry.text())
        except ValueError: pass

        if os.name == 'nt':
            self.vid = cv2.VideoCapture(cam_idx, cv2.CAP_DSHOW)
        else:
            self.vid = cv2.VideoCapture(cam_idx)
        if not self.vid.isOpened():
            self.vid = cv2.VideoCapture(cam_idx)

        if self.vid.isOpened():
            self._placeholder.hide()
            self.start_cam_btn.setText("Close Camera")
            self.btn_record.setEnabled(True)
            self._set_status(f"Camera {cam_idx} active.", CYAN)
        else:
            self._set_status(f"Failed to open camera {cam_idx}.", RED_REC)
            self.vid = None

    # ── Recording state machine (UNCHANGED logic) ──────────────────────────────
    def start_recording(self):
        if not self.vid or not self.vid.isOpened():
            QMessageBox.critical(self, "Error", "Camera is not opened.")
            return
        self.word = self.word_entry.text().strip()
        if not self.word:
            QMessageBox.critical(self, "Error", "Please enter a word label.")
            return
        try:
            self.num_samples = int(self.samples_entry.text())
            self.frames_per_sample = int(self.frames_entry.text())
        except ValueError:
            QMessageBox.critical(self, "Error", "Samples and Frames must be integers.")
            return

        self.btn_record.setEnabled(False)
        self.btn_stop.setEnabled(True)
        self.is_recording = True
        self.current_sample = 0
        self.countdown = 3
        self.wait_timer = 0
        self.last_time = time.time()
        self._update_dots()
        self._rec_dot.set_active(True)
        self._rec_label.setText("RECORDING")
        self._rec_label.setStyleSheet(f"color: {RED_REC};")
        self._stat_word.setText(self.word)

    def stop_recording(self):
        self.is_recording = False
        self.btn_record.setEnabled(True)
        self.btn_stop.setEnabled(False)
        self._rec_dot.set_active(False)
        self._rec_label.setText("IDLE")
        self._rec_label.setStyleSheet(f"color: {GREY};")
        self._set_status("Recording stopped.", AMBER)
        self.frame_progress.set_value(0.0)
        self._stat_phase.setText("—")
        self._stat_frame.setText("—")
        self._stat_sample.setText("—")

    def save_sample_task(self, frames, seq, sample_idx):
        sample_name = f"sample_{int(time.time())}"
        WEBCAM_DATA_DIR = "webcam_data"
        WEBCAM_OUT_DIR = "processed_webcam_data"
        raw_root  = Path(WEBCAM_DATA_DIR) / self.word
        land_root = Path(WEBCAM_OUT_DIR) / "landmarks" / self.word
        prev_root = Path(WEBCAM_OUT_DIR) / "preview" / self.word
        raw_root.mkdir(parents=True, exist_ok=True)
        land_root.mkdir(parents=True, exist_ok=True)
        prev_root.mkdir(parents=True, exist_ok=True)
        raw_vid_path = raw_root / f"{sample_name}.mp4"
        writer = cv2.VideoWriter(str(raw_vid_path), cv2.VideoWriter_fourcc(*"mp4v"), FPS, (WIDTH, HEIGHT))
        for f in frames: writer.write(f)
        writer.release()
        seq_np = pad(np.array(seq))
        np.save(land_root / f"{sample_name}.npy", seq_np)
        reconstruct(seq_np, prev_root / f"{sample_name}.mp4")
        print(f"Saved {sample_name} for word '{self.word}'")

    def save_sample(self):
        frames_copy = list(self.frames)
        seq_copy = list(self.seq)
        threading.Thread(target=self.save_sample_task, args=(frames_copy, seq_copy, self.current_sample)).start()

        self.current_sample += 1
        self._update_dots(done=self.current_sample)

        if self.current_sample < self.num_samples:
            self.wait_timer = 1
            self.last_time = time.time()
            self._set_status(f"Sample {self.current_sample}/{self.num_samples} saved. Waiting…", AMBER)
        else:
            self.is_recording = False
            self.btn_record.setEnabled(True)
            self.btn_stop.setEnabled(False)
            self._rec_dot.set_active(False)
            self._rec_label.setText("DONE")
            self._rec_label.setStyleSheet(f"color: {GREEN_OK};")
            self._set_status(f"Collection complete — {self.num_samples} samples captured.", GREEN_OK)
            self.frame_progress.set_value(1.0)

    def _update_dots(self, done=0):
        n = min(self.num_samples, 10)
        for i, dot in enumerate(self._sample_dots):
            if i < n:
                dot.show()
                if i < done:
                    dot.setText("●")
                    dot.setStyleSheet(f"color: {GREEN_OK}; border:none; background:transparent;")
                else:
                    dot.setText("○")
                    dot.setStyleSheet(f"color: {BORDER}; border:none; background:transparent;")
            else:
                dot.hide()

    def _set_status(self, text, color=None):
        self.status_label.setText(text)
        c = color or GREY
        self.status_label.setStyleSheet(f"color: {c};")

    # ── Frame display ──────────────────────────────────────────────────────────
    def set_image(self, frame):
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        h, w, ch = rgb.shape
        q_img = QImage(rgb.data, w, h, ch * w, QImage.Format_RGB888)
        self.video_label.setPixmap(QPixmap.fromImage(q_img))

    def update_frame(self):
        if self.vid is None or not self.vid.isOpened():
            return
        ret, frame = self.vid.read()
        if not ret:
            return
        frame = cv2.resize(frame, (WIDTH, HEIGHT))

        if self.is_recording:
            if self.countdown > 0:
                if time.time() - self.last_time >= 1.0:
                    self.countdown -= 1
                    self.last_time = time.time()

                # Overlay
                display = frame.copy()
                overlay = display.copy()
                cv2.rectangle(overlay, (0, 0), (WIDTH, HEIGHT), (10, 13, 20), -1)
                cv2.addWeighted(overlay, 0.45, display, 0.55, 0, display)

                num_text = str(self.countdown) if self.countdown > 0 else "GO"
                color = (255, 179, 64) if self.countdown > 0 else (0, 230, 118)
                (tw, th), _ = cv2.getTextSize(num_text, cv2.FONT_HERSHEY_SIMPLEX, 4, 6)
                cx, cy = WIDTH // 2 - tw // 2, HEIGHT // 2 + th // 2
                cv2.putText(display, num_text, (cx, cy), cv2.FONT_HERSHEY_SIMPLEX, 4, color, 6, cv2.LINE_AA)
                label = f"SAMPLE {self.current_sample + 1} OF {self.num_samples}  ·  {self.word}"
                (lw, _), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
                cv2.putText(display, label, (WIDTH // 2 - lw // 2, HEIGHT - 24),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (107, 122, 153), 1, cv2.LINE_AA)

                self.set_image(display)
                self._stat_phase.setText("COUNTDOWN")
                self._stat_phase.setStyleSheet(f"color: {AMBER};")
                self._stat_sample.setText(f"{self.current_sample + 1}/{self.num_samples}")

                if self.countdown == 0:
                    self.frames = []
                    self.seq = []
                    self._stat_phase.setText("CAPTURING")
                    self._stat_phase.setStyleSheet(f"color: {RED_REC};")

            elif self.wait_timer > 0:
                if time.time() - self.last_time >= 1.0:
                    self.wait_timer -= 1
                    self.last_time = time.time()
                    if self.wait_timer == 0:
                        self.countdown = 3

                display = frame.copy()
                cv2.rectangle(display, (0, 0), (WIDTH, 50), (10, 13, 20), -1)
                cv2.putText(display, "PROCESSING  ·  NEXT SAMPLE SOON",
                            (18, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 179, 64), 1, cv2.LINE_AA)
                self.set_image(display)
                self._stat_phase.setText("PROCESSING")
                self._stat_phase.setStyleSheet(f"color: {AMBER};")

            else:
                self.frames.append(frame)
                rgb_for_mp = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                res = self.holo.process(rgb_for_mp)
                self.seq.append(extract(res))

                n = len(self.frames)
                progress = n / self.frames_per_sample
                self.frame_progress.set_value(progress)
                self._stat_frame.setText(f"{n}/{self.frames_per_sample}")
                self._stat_frame.setStyleSheet(f"color: {CYAN};")
                self._stat_sample.setText(f"{self.current_sample + 1}/{self.num_samples}")

                display = frame.copy()
                # REC badge top-left
                cv2.rectangle(display, (0, 0), (200, 44), (10, 13, 20), -1)
                cv2.circle(display, (18, 22), 7, (255, 59, 59), -1)
                cv2.putText(display, f"REC  {n:03d}/{self.frames_per_sample}",
                            (32, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (237, 242, 247), 1, cv2.LINE_AA)
                # Word badge top-right
                badge = self.word
                (bw, _), _ = cv2.getTextSize(badge, cv2.FONT_HERSHEY_SIMPLEX, 0.65, 2)
                cv2.rectangle(display, (WIDTH - bw - 24, 0), (WIDTH, 44), (10, 13, 20), -1)
                cv2.putText(display, badge, (WIDTH - bw - 12, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 212, 255), 2, cv2.LINE_AA)
                self.set_image(display)

                if n >= self.frames_per_sample:
                    self.save_sample()
        else:
            # Idle passthrough with subtle label
            display = frame.copy()
            cv2.rectangle(display, (0, HEIGHT - 34), (WIDTH, HEIGHT), (10, 13, 20), -1)
            cv2.putText(display, "PREVIEW  ·  IDLE",
                        (14, HEIGHT - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (30, 42, 64), 1, cv2.LINE_AA)
            self.set_image(display)

    def closeEvent(self, event):
        if self.vid is not None:
            self.vid.release()
        self.holo.close()
        event.accept()


def main():
    app = QApplication.instance()
    if not app:
        app = QApplication(sys.argv)
    window = App()
    window.show()
    app.exec()

if __name__ == '__main__':
    main()

Saved sample_1782133377 for word 'HELLO'
Saved sample_1782133409 for word 'yikes'
